# Stage 0 — 数据构建与 Table 0

> 对应 `method.md` §5、§13.1。业务逻辑在 `sparc/data/`，本 notebook 只做调用与可视化。

**顺序纪律**：NP-purge 必须先于骨架家族与划分。否则骨架家族会把「查询分子自己」
当成记忆库邻居并进家族，让后面的断言看起来通过、实际泄漏。


In [ ]:
# 让 notebook 能 import sparc（无需 pip install -e .）
import sys, json
from pathlib import Path
CODE_ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))

from sparc.common import load_experiment_config
cfg = load_experiment_config(stage='notebook', run_name='interactive')
print('冻结配置指纹：'); print(json.dumps(cfg.freeze_manifest(), indent=1))


## 1. NPASS 规模复核（事实 H）

不做单位换算的口径应精确复现 70 / 25 / 7314 / 4075。


In [ ]:
from collections import defaultdict
from sparc.data.npass import NPASSLoader, ALLOWED_TARGET_TYPES

loader = NPASSLoader(cfg.paths.get('npass_dir'), cfg.hparams.data.csv_field_size_limit)
target_meta = loader.load_targets()
structures = loader.load_structures()

raw = defaultdict(set)
for row in loader.iter_activities('IC50'):
    s = structures.get(row.get('np_id', ''))
    t = row.get('target_id', '')
    if s and t:
        raw[t].add(s['inchikey'])

ge50 = [t for t in raw if target_meta.get(t, {}).get('target_type') in ALLOWED_TARGET_TYPES and len(raw[t]) >= 50]
ge100 = [t for t in ge50 if len(raw[t]) >= 100]
print(f'>=50 化合物靶点 = {len(ge50)}（规范 70）')
print(f'>=100 化合物靶点 = {len(ge100)}（规范 25）')
print(f'(靶点,化合物) 对 = {sum(len(raw[t]) for t in ge50)}（规范 7314）')
print(f'唯一天然产物   = {len(set().union(*[raw[t] for t in ge50]))}（规范 4075）')


## 2. 事实 D / E：靶点间化合物重叠

**只报告，不用于 R5 合并** —— R5 只按酶名合并。按重叠合并会把 CYP 家族的
7 个不同的酶压成 1 个，实测让存活靶点从 31 掉到 13。


In [ ]:
from sparc.data.npass import compound_overlap_report, build_ortholog_groups

rows = compound_overlap_report(target_meta, raw, min_compounds=20, min_overlap_frac=0.15, same_enzyme_only=True)
for r in rows[:8]:
    print(f"{r['name_a'][:26]:26s} {r['organism_a'][:20]:20s} n={r['n_a']:4d} | "
          f"{r['organism_b'][:20]:20s} n={r['n_b']:4d} | 交集 {r['intersection']:3d} = {100*r['frac_of_smaller']:.1f}%")

og = build_ortholog_groups(target_meta, raw)
for label, ids in [('AChE', ['NPT204','NPT66','NPT2668']), ('COX-1', ['NPT324','NPT30']), ('酪氨酸酶', ['NPT43','NPT741'])]:
    same = len({og[t] for t in ids if t in og}) == 1
    print(f'{label:8s} 同组 = {same}')


## 3. NP 黑名单与事实 B（79% 查询自检索面）


In [ ]:
from sparc.data.blacklist import NaturalProductBlacklist

blacklist = NaturalProductBlacklist.build(
    cfg.paths.file('coconut_lite', 'coconut_dir'),
    cfg.paths.file('lotus_gz', 'lotus_dir'),
)
print(json.dumps(blacklist.summary(), indent=1, ensure_ascii=False))

query_keys = set().union(*[raw[t] for t in ge50])
info = loader.load_general_info()
chembl = {v['inchikey']: v['chembl_id'] for v in info.values() if v.get('inchikey') and v.get('chembl_id')}
n = len(query_keys)
for label, count in [('在 COCONUT∪LOTUS', sum(1 for k in query_keys if k in blacklist.full_keys)),
                     ('自带 ChEMBL ID', sum(1 for k in query_keys if chembl.get(k))),
                     ('标为糖苷', sum(1 for k in query_keys if blacklist.is_glycoside(k)))]:
    print(f'{label:16s} {count:5d} / {n} = {100*count/n:.1f}%')


## 4. 完整管线（需要 RDKit 与已解包的 ChEMBL 37）

```bash
python scripts/run_s0_data.py --run-name s0_v1
```

完成后读回产物看 Table 0 与 Stage 0 闸门。


In [ ]:
report_path = cfg.paths.stage_outputs('s0_data') / 's0_v1_report.json'
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('NP-purge:', json.dumps(report['np_purge'], indent=1, ensure_ascii=False)[:600])
    print('Stage 0 闸门:', report['stage0_gate'])
else:
    print('尚未跑完整 Stage 0。dry-run 可在本机执行：')
    print('  python scripts/run_s0_data.py --dry-run')
